# 3D U-Net Training — Tumor Segmentation
**LUMIERE Dataset | PyTorch | GPU**

## Setup
1. Upload `data/training/` folder to Google Drive at: `MyDrive/R_project/data/training/`
2. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
3. Run all cells

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import struct, json
from google.colab import drive

drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# R torch saves in safetensors format — this loader reads it
DTYPE_MAP = {'F16': np.float16, 'F32': np.float32, 'F64': np.float64,
             'U8': np.uint8, 'I8': np.int8, 'I16': np.int16,
             'I32': np.int32, 'I64': np.int64}

def load_r_tensor(path):
    with open(path, 'rb') as f:
        header_len = struct.unpack('<Q', f.read(8))[0]
        header = json.loads(f.read(header_len))
        tensor_key = [k for k in header if k != '__metadata__'][0]
        info = header[tensor_key]
        shape = info['shape']
        np_dtype = DTYPE_MAP[info['dtype']]
        data_start = info['data_offsets'][0]
        data_end = info['data_offsets'][1]
        f.seek(8 + header_len + data_start)
        data = np.frombuffer(f.read(data_end - data_start), dtype=np_dtype)
        return torch.from_numpy(data.reshape(shape).copy())

# Quick test
test_path = Path('/content/drive/MyDrive/R_project/data/training/mri/Patient-001_week-000-1.pt')
if test_path.exists():
    t = load_r_tensor(test_path)
    print(f'Test load: shape={list(t.shape)}, dtype={t.dtype}')
else:
    print('Upload data/training/ to MyDrive/R_project/data/training/ first!')

In [ ]:
# === CONFIGURATION ===
DATA_DIR = Path('/content/drive/MyDrive/R_project/data/training')
MODEL_SAVE_DIR = Path('/content/drive/MyDrive/R_project/models')
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 2        # Increase to 4 if GPU has 16GB+ VRAM
EPOCHS = 50
LEARNING_RATE = 1e-4
N_CLASSES = 4         # background, necrosis, edema, enhancing
IN_CHANNELS = 4       # T1, T1c, T2, FLAIR
BASE_FILTERS = 16
TRAIN_SPLIT = 0.8
PATIENCE = 10

## Load Data

In [ ]:
manifest = pd.read_csv(DATA_DIR / 'manifest.csv')
print(f'Total samples: {len(manifest)}')
print(f'Patients: {manifest.patient_id.nunique()}')

# Filter out samples with no tumor
manifest = manifest[manifest.tumor_voxels > 0].reset_index(drop=True)
print(f'Samples with tumor: {len(manifest)}')
print()
print(manifest.seg_labels.value_counts())

In [ ]:
# Patient-level split (no leakage)
np.random.seed(42)
patient_ids = manifest.patient_id.unique()
np.random.shuffle(patient_ids)
n_train = int(len(patient_ids) * TRAIN_SPLIT)
train_patients = set(patient_ids[:n_train])
val_patients = set(patient_ids[n_train:])

train_manifest = manifest[manifest.patient_id.isin(train_patients)].reset_index(drop=True)
val_manifest = manifest[manifest.patient_id.isin(val_patients)].reset_index(drop=True)

print(f'Train: {len(train_manifest)} samples from {len(train_patients)} patients')
print(f'Val:   {len(val_manifest)} samples from {len(val_patients)} patients')

## Dataset & DataLoader

In [ ]:
class TumorDataset(Dataset):
    def __init__(self, manifest_df, data_dir, augment=False):
        self.manifest = manifest_df
        self.data_dir = Path(data_dir)
        self.augment = augment

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        row = self.manifest.iloc[idx]
        mri = load_r_tensor(self.data_dir / row.mri_file).float()
        seg = load_r_tensor(self.data_dir / row.seg_file).long()

        if self.augment:
            if np.random.random() > 0.5:
                mri = torch.flip(mri, [1])
                seg = torch.flip(seg, [0])
            if np.random.random() > 0.5:
                mri = torch.flip(mri, [2])
                seg = torch.flip(seg, [1])
            if np.random.random() > 0.5:
                mri = torch.flip(mri, [3])
                seg = torch.flip(seg, [2])
            scale = 0.9 + np.random.random() * 0.2
            mri = mri * scale

        return mri, seg

print('TumorDataset class defined. Run next cell to copy data locally and build DataLoaders.')

In [ ]:
import shutil

LOCAL_DIR = Path('/content/training_data')
if not LOCAL_DIR.exists():
    print('Copying data from Drive to local disk (one-time, ~500MB)...')
    shutil.copytree(DATA_DIR, LOCAL_DIR)
    print(f'Done! Copied to {LOCAL_DIR}')
else:
    print('Local copy already exists')

train_ds = TumorDataset(train_manifest, LOCAL_DIR, augment=True)
val_ds = TumorDataset(val_manifest, LOCAL_DIR, augment=False)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'DataLoaders ready: {len(train_dl)} train batches, {len(val_dl)} val batches')

## 3D U-Net Model

In [ ]:
class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm3d(out_ch)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x

class EncoderBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv_block = ConvBlock3D(in_ch, out_ch)
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)

    def forward(self, x):
        features = self.conv_block(x)
        pooled = self.pool(features)
        return features, pooled

class DecoderBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.upconv = nn.ConvTranspose3d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv_block = ConvBlock3D(out_ch * 2, out_ch)

    def forward(self, x, skip):
        x = self.upconv(x)
        # Pad if sizes don't match
        diff_d = skip.size(2) - x.size(2)
        diff_h = skip.size(3) - x.size(3)
        diff_w = skip.size(4) - x.size(4)
        x = F.pad(x, [
            diff_w // 2, diff_w - diff_w // 2,
            diff_h // 2, diff_h - diff_h // 2,
            diff_d // 2, diff_d - diff_d // 2
        ])
        x = torch.cat([x, skip], dim=1)
        return self.conv_block(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=4, n_classes=4, base_filters=16):
        super().__init__()
        f = base_filters
        self.enc1 = EncoderBlock3D(in_channels, f)
        self.enc2 = EncoderBlock3D(f, f * 2)
        self.enc3 = EncoderBlock3D(f * 2, f * 4)
        self.enc4 = EncoderBlock3D(f * 4, f * 8)
        self.bottleneck = ConvBlock3D(f * 8, f * 16)
        self.dec4 = DecoderBlock3D(f * 16, f * 8)
        self.dec3 = DecoderBlock3D(f * 8, f * 4)
        self.dec2 = DecoderBlock3D(f * 4, f * 2)
        self.dec1 = DecoderBlock3D(f * 2, f)
        self.final_conv = nn.Conv3d(f, n_classes, kernel_size=1)

    def forward(self, x):
        f1, p1 = self.enc1(x)
        f2, p2 = self.enc2(p1)
        f3, p3 = self.enc3(p2)
        f4, p4 = self.enc4(p3)
        b = self.bottleneck(p4)
        d4 = self.dec4(b, f4)
        d3 = self.dec3(d4, f3)
        d2 = self.dec2(d3, f2)
        d1 = self.dec1(d2, f1)
        return self.final_conv(d1)

model = UNet3D(IN_CHANNELS, N_CLASSES, BASE_FILTERS).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'3D U-Net parameters: {n_params:,}')

# Verify forward pass
with torch.no_grad():
    dummy = torch.randn(1, 4, 48, 48, 48).to(device)
    out = model(dummy)
    print(f'Forward pass: {list(dummy.shape)} → {list(out.shape)}')

## Loss Functions

In [ ]:
def dice_loss(pred, target, smooth=1.0):
    pred_soft = F.softmax(pred, dim=1)
    n_classes = pred.size(1)
    total_loss = 0.0

    for c in range(n_classes):
        pred_c = pred_soft[:, c]
        target_c = (target == c).float()
        intersection = (pred_c * target_c).sum()
        union_val = pred_c.sum() + target_c.sum()
        dice = (2 * intersection + smooth) / (union_val + smooth)
        total_loss += (1 - dice)

    return total_loss / n_classes

def combined_loss(pred, target):
    ce = F.cross_entropy(pred, target)
    dl = dice_loss(pred, target)
    return 0.5 * ce + 0.5 * dl

def compute_dice_score(pred, target, n_classes=4):
    pred_class = pred.argmax(dim=1)
    scores = []
    for c in range(1, n_classes):  # skip background
        pred_c = (pred_class == c).float()
        target_c = (target == c).float()
        intersection = (pred_c * target_c).sum().item()
        union_val = pred_c.sum().item() + target_c.sum().item()
        dice = 2 * intersection / union_val if union_val > 0 else 1.0
        scores.append(dice)
    return np.mean(scores)

## Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

best_val_dice = 0
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'val_dice': []}
patience_counter = 0

print('Starting training...\n')

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    for batch_idx, (mri, seg) in enumerate(train_dl):
        mri, seg = mri.to(device), seg.to(device)
        optimizer.zero_grad()
        pred = model(mri)
        loss = combined_loss(pred, seg)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())
        if batch_idx % 20 == 0:
            print(f'  Epoch {epoch} [{batch_idx}/{len(train_dl)}] loss={loss.item():.4f}', end='\r')

    avg_train_loss = np.mean(train_losses)

    model.eval()
    val_losses = []
    val_dices = []
    with torch.no_grad():
        for mri, seg in val_dl:
            mri, seg = mri.to(device), seg.to(device)
            pred = model(mri)
            val_losses.append(combined_loss(pred, seg).item())
            val_dices.append(compute_dice_score(pred, seg, N_CLASSES))

    avg_val_loss = np.mean(val_losses)
    avg_val_dice = np.mean(val_dices)
    scheduler.step()

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_dice'].append(avg_val_dice)

    msg = f'Epoch {epoch:02d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Dice: {avg_val_dice:.4f}'

    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), MODEL_SAVE_DIR / 'unet_3d_trained.pt')
        msg += ' *BEST*'
        patience_counter = 0
    else:
        patience_counter += 1

    print(msg)

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\n{"="*50}')
print(f'Training Complete')
print(f'Best Val Dice: {best_val_dice:.4f}')
print(f'Best Val Loss: {best_val_loss:.4f}')
print(f'Model saved: {MODEL_SAVE_DIR / "unet_3d_trained.pt"}')
print(f'{"="*50}')

## Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs, history['train_loss'], 'b-', linewidth=1.5, label='Train')
ax1.plot(epochs, history['val_loss'], 'r-', linewidth=1.5, label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history['val_dice'], 'g-', linewidth=1.5)
ax2.axhline(y=best_val_dice, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Dice Score')
ax2.set_title(f'Validation Dice (Best: {best_val_dice:.4f})', fontweight='bold')
ax2.grid(True, alpha=0.3)

fig.suptitle('3D U-Net Training on LUMIERE Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(MODEL_SAVE_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## Evaluate on Validation Set

In [ ]:
# Load best model
model.load_state_dict(torch.load(MODEL_SAVE_DIR / 'unet_3d_trained.pt', weights_only=True))
model.eval()

class_names = ['Necrosis', 'Edema', 'Enhancing']
per_class_dice = []

with torch.no_grad():
    for i in range(len(val_manifest)):
        row = val_manifest.iloc[i]
        mri = load_r_tensor(DATA_DIR / row.mri_file).float().unsqueeze(0).to(device)
        seg = load_r_tensor(DATA_DIR / row.seg_file).long().to(device)
        pred = model(mri)
        pred_class = pred.argmax(dim=1).squeeze(0)

        scores = []
        for c in range(1, 4):
            pred_c = (pred_class == c).float()
            target_c = (seg == c).float()
            intersection = (pred_c * target_c).sum().item()
            union_val = pred_c.sum().item() + target_c.sum().item()
            dice = 2 * intersection / union_val if union_val > 0 else float('nan')
            scores.append(dice)
        per_class_dice.append(scores)

per_class_dice = np.array(per_class_dice)

print('Per-Class Dice Scores (Validation Set):')
for c in range(3):
    scores = per_class_dice[:, c]
    valid = scores[~np.isnan(scores)]
    print(f'  {class_names[c]}: {valid.mean():.4f} +/- {valid.std():.4f} (n={len(valid)})')
print(f'  Overall: {np.nanmean(per_class_dice):.4f}')

## Visual Results

In [ ]:
n_show = min(4, len(val_manifest))
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

seg_cmap = plt.cm.colors.ListedColormap(['black', '#e74c3c', '#f39c12', '#2ecc71'])

with torch.no_grad():
    for i in range(n_show):
        row = val_manifest.iloc[i]
        mri = load_r_tensor(DATA_DIR / row.mri_file).float().unsqueeze(0).to(device)
        seg_gt = load_r_tensor(DATA_DIR / row.seg_file).numpy()
        pred = model(mri)
        seg_pred = pred.argmax(dim=1).squeeze(0).cpu().numpy()
        mri_np = mri.squeeze(0).cpu().numpy()

        z = seg_gt.shape[2] // 2

        axes[i, 0].imshow(mri_np[0, :, :, z], cmap='gray')
        axes[i, 0].set_title(f'{row.patient_id} — FLAIR')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(seg_gt[:, :, z], cmap=seg_cmap, vmin=0, vmax=3)
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(seg_pred[:, :, z], cmap=seg_cmap, vmin=0, vmax=3)
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')

fig.suptitle('3D U-Net: Ground Truth vs Prediction (Axial Slice)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(MODEL_SAVE_DIR / 'visual_results.png'), dpi=150, bbox_inches='tight')
plt.show()

## Save Results & Export for R

In [ ]:
# Save results CSV
results = pd.DataFrame({
    'metric': ['Best Val Dice', 'Best Val Loss'] +
              [f'Dice - {n}' for n in class_names] +
              ['Overall Dice', 'Epochs Trained', 'Parameters'],
    'value': [
        f'{best_val_dice:.4f}',
        f'{best_val_loss:.4f}',
        *[f'{np.nanmean(per_class_dice[:, c]):.4f}' for c in range(3)],
        f'{np.nanmean(per_class_dice):.4f}',
        str(len(history['train_loss'])),
        f'{n_params:,}'
    ]
})
print(results.to_string(index=False))
results.to_csv(MODEL_SAVE_DIR / 'unet_results.csv', index=False)

# Save full model (architecture + weights) for R compatibility
# TorchScript export — loadable from both Python and R
model.eval()
model_cpu = model.cpu()
scripted = torch.jit.trace(model_cpu, torch.randn(1, 4, 48, 48, 48))
scripted.save(str(MODEL_SAVE_DIR / 'unet_3d_trained_jit.pt'))
print(f'\nTorchScript model saved: {MODEL_SAVE_DIR / "unet_3d_trained_jit.pt"}')
print('This file is loadable from R using torch::jit_load()')

# Also save state_dict
torch.save(model_cpu.state_dict(), MODEL_SAVE_DIR / 'unet_3d_trained.pt')

print(f'\nDone! Download these files to your local project:')
print(f'  models/unet_3d_trained_jit.pt  (for R inference)')
print(f'  models/unet_3d_trained.pt       (state_dict for Python)')
print(f'  models/unet_results.csv')
print(f'  models/training_curves.png')
print(f'  models/visual_results.png')